In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
model= ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [ ]:
###summerization

In [19]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
agent = create_agent(model = model,checkpointer = InMemorySaver(),  middleware = [SummarizationMiddleware(
    model = model, #middleware runs before calling the model 
    trigger = ("tokens", 100), # after 100 token invoke summarization middleware
    keep = ("messages", 1) #N=1
'''
When summarization triggers, middleware rewrites history into:

One synthetic summary message (lc_source: "summarization"), type is HumanMessage
Plus last N original messages where N = keep
'''
    
)])

config = {"configurable" : {"thread_id":1}}

In [20]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke({"messages": [
    HumanMessage("My name is Nitin"),
    AIMessage("Good to know you name. Hi Nitin how are you?"),
    HumanMessage("Can you tell me about Ironman"),
    AIMessage("Iron Man is a Marvel superhero and genius billionaire, Tony Stark, who uses advanced armored suits for flight, strength, and energy repulsor rays. Created by Stan Lee, he debuted in 1963 and became a cornerstone of the Marvel Cinematic Universe (MCU), famously played by Robert Downey Jr. starting in 2008"),
    HumanMessage("according to you who is more powerful than ironman?"),
    AIMessage('1. Thor: Frequently cited as the most powerful Avenger. In both comics and films, Thor’s godly physiology and control over lightning allow him to withstand Iron Man’s strongest attacks.2.Captain Marvel (Carol Danvers): Described as having power levels similar to a "walking sun," her binary form can destroy planets—a feat far beyond the capabilities of standard Iron Man suits.'),
    
]}, config = config)

response

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about characters within the Marvel universe who possess greater power levels than Iron Man.\n\n## SUMMARY\n*   User: Nitin.\n*   The conversation has established the persona of Iron Man (Tony Stark) as a genius billionaire and armored superhero.\n*   The current discussion is focused on comparing Iron Man's power levels against other Marvel entities.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nProvide a list or analysis of Marvel characters who are canonically more powerful than Iron Man, explaining the reasoning for these power level differences (e.g., cosmic entities, magic-wielders, or god-tier characters).", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='99707968-7d69-4baf-a311-45f001214690'),
  AIMessage(content='1. Thor: Frequently cited as the most powerful Avenger. In both comics and films, Thor’s godly physiology and control o

In [18]:
response = agent.invoke({"messages": [
    HumanMessage("What is my name?"),
    
]}, config = config)

response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user, Nitin, is exploring the character of Iron Man (Tony Stark), specifically seeking comparisons regarding his power level relative to other characters in the Marvel universe.\n\n## SUMMARY\n*   **User Name:** Nitin.\n*   **Character Context:** Iron Man is a genius billionaire created by Stan Lee in 1963, central to the MCU and portrayed by Robert Downey Jr.\n*   **Power Comparison:** Thor (godly physiology/lightning control) and Captain Marvel (binary form/planet-destroying capabilities) have been identified as characters more powerful than Iron Man.\n*   **Rejected Options:** None.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nAddress any further questions Nitin may have regarding power rankings, specific Iron Man armor iterations, or other Marvel-related inquiries.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='c7d11892-e89a-41ec-a74c-5c56c6ee412f'

In [21]:
###trim/delete older messages

In [36]:
from langchain.messages import RemoveMessage
from langchain.agents import AgentState
from langchain.tools import ToolRuntime
from langchain.agents.middleware import before_agent
from typing import Any
@before_agent
def remove_older_messages(state: AgentState, runtime: ToolRuntime)-> dict[str, Any] | None:
    messages = state['messages']
    if len(messages) < 3: 
        return None #dont delete <=2 messages

    remove_messages = messages[:2] #remove 1st two message
    return {"messages" : [RemoveMessage(id = m.id) for m in remove_messages]}
    

In [37]:

agent = create_agent(model = model,checkpointer = InMemorySaver(),  middleware = [remove_older_messages])

config = {"configurable" : {"thread_id":1}}

In [38]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke({"messages": [
    HumanMessage("My name is Nitin"),
    AIMessage("Good to know you name. Hi Nitin how are you?"),
    HumanMessage("Can you tell me about Ironman"),
    AIMessage("Iron Man is a Marvel superhero and genius billionaire, Tony Stark, who uses advanced armored suits for flight, strength, and energy repulsor rays. Created by Stan Lee, he debuted in 1963 and became a cornerstone of the Marvel Cinematic Universe (MCU), famously played by Robert Downey Jr. starting in 2008"),
    HumanMessage("according to you who is more powerful than ironman?"),
    AIMessage('1. Thor: Frequently cited as the most powerful Avenger. In both comics and films, Thor’s godly physiology and control over lightning allow him to withstand Iron Man’s strongest attacks.2.Captain Marvel (Carol Danvers): Described as having power levels similar to a "walking sun," her binary form can destroy planets—a feat far beyond the capabilities of standard Iron Man suits.'),
    
]}, config = config)

response

{'messages': [HumanMessage(content='Can you tell me about Ironman', additional_kwargs={}, response_metadata={}, id='133a0503-f23d-4d3c-9fd1-e16d60abe898'),
  AIMessage(content='Iron Man is a Marvel superhero and genius billionaire, Tony Stark, who uses advanced armored suits for flight, strength, and energy repulsor rays. Created by Stan Lee, he debuted in 1963 and became a cornerstone of the Marvel Cinematic Universe (MCU), famously played by Robert Downey Jr. starting in 2008', additional_kwargs={}, response_metadata={}, id='20a0fdae-a4dd-4c3b-b2ed-0f0f66c911c6', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='according to you who is more powerful than ironman?', additional_kwargs={}, response_metadata={}, id='edc856d7-f3ed-4cdd-9c3f-684baf2979a8'),
  AIMessage(content='1. Thor: Frequently cited as the most powerful Avenger. In both comics and films, Thor’s godly physiology and control over lightning allow him to withstand Iron Man’s strongest attacks.2.Captain Marvel 

In [39]:
response = agent.invoke({"messages": [
    HumanMessage("What is my name?"),
    
]}, config = config)

response #in trim message we can see agent dont know my name: memory loss can happen

{'messages': [HumanMessage(content='according to you who is more powerful than ironman?', additional_kwargs={}, response_metadata={}, id='edc856d7-f3ed-4cdd-9c3f-684baf2979a8'),
  AIMessage(content='1. Thor: Frequently cited as the most powerful Avenger. In both comics and films, Thor’s godly physiology and control over lightning allow him to withstand Iron Man’s strongest attacks.2.Captain Marvel (Carol Danvers): Described as having power levels similar to a "walking sun," her binary form can destroy planets—a feat far beyond the capabilities of standard Iron Man suits.', additional_kwargs={}, response_metadata={}, id='989b0a5b-97bc-4b00-9cdf-8c2401657e65', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content=[{'type': 'text', 'text': '3. Scarlet Witch (Wanda Maximoff): She possesses reality-warping capabilities that can erase or alter things instantly, making standard technology (no matter how advanced) largely irrelevant.4. Thanos (with Infinity Stones): As shown in "Infinity 